# Manejo de Errores, Panic y Defer en Go

Go adopta un enfoque distintivo para el manejo de situaciones excepcionales. En lugar de excepciones tradicionales (`try-catch`), Go utiliza valores de retorno para errores esperados y un mecanismo de `panic/recover` para errores catastróficos.

## 1. El Patrón de Errores en Go

En Go, los errores son simplemente valores que implementan la interfaz `error`.

```go
type error interface {
    Error() string
}
```

El patrón estándar es retornar el error como el último valor de retorno de una función.

In [ ]:
package main

import (
    "errors"
    "fmt"
)

func validarEdad(edad int) (bool, error) {
    if edad < 0 {
        return false, errors.New("la edad no puede ser negativa")
    }
    if edad < 18 {
        return false, fmt.Errorf("edad %d es insuficiente", edad)
    }
    return true, nil
}

func main() {
    esAdulto, err := validarEdad(-5)
    if err != nil {
        fmt.Println("Error detectado:", err)
        return
    }
    fmt.Println("Es adulto:", esAdulto)
}

## 2. Sentencia `defer`

La sentencia `defer` pospone la ejecución de una función hasta que la función que la rodea retorna. Se utiliza comúnmente para tareas de limpieza (cerrar archivos, liberar mutexes, cerrar conexiones de red).

In [ ]:
func ejemploDefer() {
    defer fmt.Println("Mundo") // Se ejecutará al final
    fmt.Println("Hola")
}

// Los defers se ejecutan en orden LIFO (Last In, First Out)
func ordenDefer() {
    for i := 0; i < 3; i++ {
        defer fmt.Println(i)
    }
    // Salida: 2, 1, 0
}

## 3. `panic` y `recover`

- **`panic`**: Detiene el flujo normal de ejecución, ejecuta los `defer` y termina el programa.
- **`recover`**: Es una función integrada que recupera el control de una goroutine en pánico. Solo es útil dentro de funciones diferidas (`defer`).

In [ ]:
func protegerAccion() {
    defer func() {
        if r := recover(); r != nil {
            fmt.Println("Recuperado del pánico:", r)
        }
    }()

    fmt.Println("Iniciando algo peligroso...")
    panic("¡Algo salió muy mal!")
    fmt.Println("Esto nunca se ejecutará")
}

// REGLA: Usa panic solo para errores irrecuperables (ej. fallo de configuración inicial).

## 4. Wrapping de Errores (Go 1.13+)

Go permite añadir contexto a los errores manteniendo el error original mediante el verbo `%w` en `fmt.Errorf`.

In [ ]:
var ErrNoEncontrado = errors.New("no encontrado")

func buscarDato() error {
    return fmt.Errorf("error en la base de datos: %w", ErrNoEncontrado)
}

func main() {
    err := buscarDato()
    
    // Comprobar si el error es de un tipo específico (Errors Is)
    if errors.Is(err, ErrNoEncontrado) {
        fmt.Println("Efectivamente, no se encontró el dato")
    }

    // Extraer el error original a una estructura (Errors As)
    /*
    var pathErr *os.PathError
    if errors.As(err, &pathErr) {
        fmt.Println("Error en ruta:", pathErr.Path)
    }
    */
}

## 5. Mejores Prácticas

1. **No ignores los errores**: Siempre verifica `if err != nil`. El identificador `_` para errores debe usarse con extrema precaución.
2. **Contexto**: Añade información útil al error (`fmt.Errorf("procesando usuario %d: %w", id, err)`).
3. **Errores Centinela**: Define errores globales para casos comunes (`var ErrNotFound = errors.New("...")`).
4. **Tipos de Error Personalizados**: Crea structs que implementen `Error()` si necesitas llevar más datos.